In [1]:
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation

from stmorlet import Morlet2D1T

## 弯曲磁场中的慢磁声波波包测试信号

这里构造一个运动学/WKB型测试波包，而不是完整的MHD数值模拟。局部背景磁场初始沿 $x$ 轴，随后沿波包轨迹缓慢弯曲。波矢与局部磁场之间的夹角也缓慢变化。每个时刻都使用理想MHD慢磁声波色散关系

$$
v_{\mathrm{s}}^2(\alpha)=\frac{1}{2}\left[v_A^2+c_s^2-\sqrt{(v_A^2+c_s^2)^2-4v_A^2c_s^2\cos^2\alpha}\right],
\qquad \omega=k v_{\mathrm{s}}(\alpha),
$$

并由 $\boldsymbol{v}_g=\nabla_{\boldsymbol{k}}\omega$ 计算群速度。波包质心沿群速度积分得到，因此波包轨迹、载波传播方向以及局部磁场方向是三个可以分别检验的量。

总时间轴长于波包的有效寿命：波包只在时间轴中央生成、传播并消失，前后留白用于容纳低频Morlet核。数据轴长度同时满足波包包络范围和计划分析频段 $3$--$4\,\mathrm{mHz}$ 的Morlet有效支撑范围。

## 生成测试信号

In [2]:
# -------------------------
# 1. Sampling and MHD parameters
# -------------------------
dx = 0.4342  # Mm
dy = 0.4342  # Mm
dt = 12.0    # s

v_alfven = 0.450  # Mm/s = 450 km/s
c_sound = 0.150   # Mm/s = 150 km/s

carrier_frequency = 3.5e-3  # Hz; target carrier frequency
carrier_omega = 2.0 * np.pi * carrier_frequency
time_axis_cycles = 6.0       # total data duration in carrier periods
packet_active_cycles = 3.0   # finite packet lifetime in carrier periods
amplitude = 1.0
noise_std = 0.0  # First test is noise-free; increase later if needed.
random_seed = 20260805

final_field_angle = np.deg2rad(25.0)
final_obliquity = np.deg2rad(10.0)
margin_sigma = 2.0

# Parameters used by the first CWT test.  The support calculation below
# follows stmorlet._diagnostics.warn_wavelet_sampling_limits.
analysis_frequency_min = 3.0e-3  # Hz
analysis_frequency_max = 4.0e-3  # Hz
morlet_k0 = 6.0
morlet_omega0 = 6.0
morlet_epsilon = 1.0
support_margin = 1.10


def smoothstep01(value):
    """Smoothly map [0, 1] to [0, 1] with zero endpoint slopes."""
    return value**2 * (3.0 - 2.0 * value)


def next_multiple(value, multiple=8):
    """Round a positive integer upward for FFT-friendly array sizes."""
    return int(np.ceil(value / multiple) * multiple)


def slow_mode_speed_and_angular_derivative(alpha, v_a, c_s):
    """Return slow-mode phase speed and d(v_phase)/d(alpha)."""
    speed_sum = v_a**2 + c_s**2
    discriminant = np.sqrt(
        speed_sum**2
        - 4.0 * v_a**2 * c_s**2 * np.cos(alpha) ** 2
    )
    v_phase = np.sqrt(0.5 * (speed_sum - discriminant))
    dv_dalpha = -(
        v_a**2 * c_s**2 * np.sin(alpha) * np.cos(alpha)
    ) / (v_phase * discriminant)
    return v_phase, dv_dalpha


# -------------------------
# 2. Local dispersion and the packet-centre trajectory
# -------------------------
requested_duration = time_axis_cycles / carrier_frequency
nt = next_multiple(int(np.ceil(requested_duration / dt)))
t = np.arange(nt, dtype=float) * dt
normalized_time = t / t[-1]

packet_active_duration = packet_active_cycles / carrier_frequency
packet_start_time = 0.5 * (t[-1] - packet_active_duration)
packet_end_time = packet_start_time + packet_active_duration
packet_progress = np.clip(
    (t - packet_start_time) / packet_active_duration, 0.0, 1.0
)
packet_active = (t >= packet_start_time) & (t <= packet_end_time)
bend_fraction = smoothstep01(packet_progress)

field_angle = final_field_angle * bend_fraction
obliquity = final_obliquity * bend_fraction
wavevector_angle = field_angle + obliquity

phase_speed, dv_dalpha = slow_mode_speed_and_angular_derivative(
    obliquity, v_alfven, c_sound
)
wavenumber = carrier_omega / phase_speed
kx_carrier = wavenumber * np.cos(wavevector_angle)
ky_carrier = wavenumber * np.sin(wavevector_angle)

# For omega = k * v_phase(alpha):
# v_group = v_phase * e_k + (d v_phase / d alpha) * e_theta.
group_velocity_x = (
    phase_speed * np.cos(wavevector_angle)
    - dv_dalpha * np.sin(wavevector_angle)
)
group_velocity_y = (
    phase_speed * np.sin(wavevector_angle)
    + dv_dalpha * np.cos(wavevector_angle)
)
group_speed = np.hypot(group_velocity_x, group_velocity_y)
group_angle = np.arctan2(group_velocity_y, group_velocity_x)

x_center_relative = np.zeros(nt, dtype=float)
y_center_relative = np.zeros(nt, dtype=float)
interval_start = np.maximum(t[:-1], packet_start_time)
interval_end = np.minimum(t[1:], packet_end_time)
active_interval = np.maximum(interval_end - interval_start, 0.0)
x_center_relative[1:] = np.cumsum(
    0.5 * (group_velocity_x[:-1] + group_velocity_x[1:]) * active_interval
)
y_center_relative[1:] = np.cumsum(
    0.5 * (group_velocity_y[:-1] + group_velocity_y[1:]) * active_interval
)

# -------------------------
# 3. Determine the data size from the trajectory and envelope widths
# -------------------------
local_wavelength = 2.0 * np.pi / wavenumber
reference_wavelength = float(np.median(local_wavelength))
sigma_parallel = 0.35 * reference_wavelength
sigma_perpendicular = 0.18 * reference_wavelength

x_projected_sigma = np.sqrt(
    (sigma_parallel * np.cos(group_angle)) ** 2
    + (sigma_perpendicular * np.sin(group_angle)) ** 2
)
y_projected_sigma = np.sqrt(
    (sigma_parallel * np.sin(group_angle)) ** 2
    + (sigma_perpendicular * np.cos(group_angle)) ** 2
)

x_signal_min = np.min(x_center_relative - margin_sigma * x_projected_sigma)
x_signal_max = np.max(x_center_relative + margin_sigma * x_projected_sigma)
y_signal_min = np.min(y_center_relative - margin_sigma * y_projected_sigma)
y_signal_max = np.max(y_center_relative + margin_sigma * y_projected_sigma)

# The smallest k and omega in the planned analysis have the largest support.
# c_sound is the maximum slow-mode phase speed, so this k_min is conservative.
analysis_omega_min = 2.0 * np.pi * analysis_frequency_min
analysis_k_min = analysis_omega_min / c_sound
maximum_wavelet_support_x = 2.0 * morlet_k0 / analysis_k_min
maximum_wavelet_support_y = 2.0 * morlet_k0 / analysis_k_min
maximum_wavelet_support_t = 2.0 * morlet_omega0 / analysis_omega_min

minimum_x_extent = support_margin * maximum_wavelet_support_x
minimum_y_extent = support_margin * maximum_wavelet_support_y
minimum_t_extent = support_margin * maximum_wavelet_support_t

x_signal_extent = x_signal_max - x_signal_min
y_signal_extent = y_signal_max - y_signal_min
x_required_extent = max(x_signal_extent, minimum_x_extent)
y_required_extent = max(y_signal_extent, minimum_y_extent)
x_required_min = x_signal_min - 0.5 * (x_required_extent - x_signal_extent)
x_required_max = x_signal_max + 0.5 * (x_required_extent - x_signal_extent)
y_required_min = y_signal_min - 0.5 * (y_required_extent - y_signal_extent)
y_required_max = y_signal_max + 0.5 * (y_required_extent - y_signal_extent)

nx_required = int(np.ceil((x_required_max - x_required_min) / dx)) + 1
ny_required = int(np.ceil((y_required_max - y_required_min) / dy)) + 1
nx = next_multiple(nx_required)
ny = next_multiple(ny_required)

x = np.arange(nx, dtype=float) * dx
y = np.arange(ny, dtype=float) * dy
x_extra = x[-1] - (x_required_max - x_required_min)
y_extra = y[-1] - (y_required_max - y_required_min)
x_center = x_center_relative - x_required_min + 0.5 * x_extra
y_center = y_center_relative - y_required_min + 0.5 * y_extra

# -------------------------
# 4. Generate the finite-lifetime curved wave packet
# -------------------------
temporal_envelope = np.zeros(nt, dtype=float)
temporal_envelope[packet_active] = (
    np.sin(np.pi * packet_progress[packet_active]) ** 4
)
phase_center_rate = (
    kx_carrier * group_velocity_x
    + ky_carrier * group_velocity_y
    - carrier_omega
)
phase_center = np.zeros(nt, dtype=float)
phase_center[1:] = np.cumsum(
    0.5 * (phase_center_rate[:-1] + phase_center_rate[1:]) * dt
)

clean_signal = np.empty((nx, ny, nt), dtype=np.float32)
for time_index in range(nt):
    delta_x = x[:, None] - x_center[time_index]
    delta_y = y[None, :] - y_center[time_index]

    cos_group = np.cos(group_angle[time_index])
    sin_group = np.sin(group_angle[time_index])
    coordinate_parallel = delta_x * cos_group + delta_y * sin_group
    coordinate_perpendicular = -delta_x * sin_group + delta_y * cos_group

    spatial_envelope = np.exp(
        -0.5 * (coordinate_parallel / sigma_parallel) ** 2
        -0.5 * (coordinate_perpendicular / sigma_perpendicular) ** 2
    )
    carrier_phase = (
        kx_carrier[time_index] * delta_x
        + ky_carrier[time_index] * delta_y
        + phase_center[time_index]
    )
    clean_signal[:, :, time_index] = (
        amplitude
        * temporal_envelope[time_index]
        * spatial_envelope
        * np.cos(carrier_phase)
    ).astype(np.float32)

rng = np.random.default_rng(random_seed)
test_signal = clean_signal.copy()
if noise_std > 0.0:
    test_signal += rng.normal(0.0, noise_std, test_signal.shape).astype(np.float32)

truth = {
    "time": t,
    "packet_active": packet_active,
    "temporal_envelope": temporal_envelope,
    "x_center": x_center,
    "y_center": y_center,
    "field_angle": field_angle,
    "obliquity": obliquity,
    "wavevector_angle": wavevector_angle,
    "group_angle": group_angle,
    "phase_speed": phase_speed,
    "group_velocity_x": group_velocity_x,
    "group_velocity_y": group_velocity_y,
    "group_speed": group_speed,
    "wavenumber": wavenumber,
    "omega": np.full(nt, carrier_omega),
    "maximum_wavelet_support": np.asarray([
        maximum_wavelet_support_x,
        maximum_wavelet_support_y,
        maximum_wavelet_support_t,
    ]),
}

dispersion_residual = np.max(
    np.abs(carrier_omega - wavenumber * phase_speed)
)
print(f"test_signal shape: {test_signal.shape}, dtype: {test_signal.dtype}")
print(f"array size: {test_signal.nbytes / 2**20:.2f} MiB")
print(f"duration: {t[-1]:.1f} s, time-axis cycles: {t[-1] * carrier_frequency:.2f}")
print(f"active packet duration: {packet_active_duration:.1f} s ({packet_active_cycles:.2f} cycles)")
print(f"wavelength range: {local_wavelength.min():.2f}--{local_wavelength.max():.2f} Mm")
print(f"phase speed range: {phase_speed.min() * 1000:.2f}--{phase_speed.max() * 1000:.2f} km/s")
print(f"group speed range: {group_speed.min() * 1000:.2f}--{group_speed.max() * 1000:.2f} km/s")
print(
    "maximum wavelet support (x, y, t): "
    f"{maximum_wavelet_support_x:.2f} Mm, "
    f"{maximum_wavelet_support_y:.2f} Mm, "
    f"{maximum_wavelet_support_t:.2f} s"
)
print(f"data extents used by stmorlet checks: {nx * dx:.2f} Mm, {ny * dy:.2f} Mm, {nt * dt:.2f} s")
print(f"maximum dispersion residual: {dispersion_residual:.3e} rad/s")

test_signal shape: (424, 248, 144), dtype: float32
array size: 57.76 MiB
duration: 1716.0 s, time-axis cycles: 6.01
active packet duration: 857.1 s (3.00 cycles)
wavelength range: 42.13--42.86 Mm
phase speed range: 147.44--150.00 km/s
group speed range: 150.00--150.29 km/s
maximum wavelet support (x, y, t): 95.49 Mm, 95.49 Mm, 636.62 s
data extents used by stmorlet checks: 184.10 Mm, 107.68 Mm, 1728.00 s
maximum dispersion residual: 3.469e-18 rad/s


In [ ]:
# Visual checks of the packet, trajectory, directions, and speeds.
peak_time_index = int(np.argmax(temporal_envelope))
maximum_amplitude_map = np.max(np.abs(test_signal), axis=2)

fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)

image = axes[0, 0].pcolormesh(
    x, y, maximum_amplitude_map.T, shading="auto", cmap="magma"
)
axes[0, 0].plot(x_center[packet_active], y_center[packet_active], color="cyan", linewidth=2, label="true centre")
axes[0, 0].set(title="Maximum absolute amplitude and true trajectory", xlabel="x [Mm]", ylabel="y [Mm]")
axes[0, 0].legend()
fig.colorbar(image, ax=axes[0, 0], label="amplitude")

image = axes[0, 1].pcolormesh(
    x, y, test_signal[:, :, peak_time_index].T, shading="auto", cmap="RdBu_r"
)
axes[0, 1].plot(x_center[peak_time_index], y_center[peak_time_index], "ko", markersize=4)
axes[0, 1].set(title=f"Signal at t = {t[peak_time_index]:.0f} s", xlabel="x [Mm]", ylabel="y [Mm]")
fig.colorbar(image, ax=axes[0, 1], label="signal")

axes[1, 0].plot(t, np.rad2deg(field_angle), label="local magnetic field")
axes[1, 0].plot(t, np.rad2deg(wavevector_angle), label="wavevector")
axes[1, 0].plot(t, np.rad2deg(group_angle), label="group velocity")
axes[1, 0].set(title="Direction evolution", xlabel="t [s]", ylabel="angle [degree]")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(t, phase_speed * 1000.0, label="phase speed")
axes[1, 1].plot(t, group_speed * 1000.0, label="group speed")
axes[1, 1].set(title="Slow-mode speeds", xlabel="t [s]", ylabel="speed [km/s]")
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 8))
im = ax.imshow(test_signal[:, :, 0], cmap='RdBu_r', extent=(y[0], y[-1], x[0], x[-1]), vmin=test_signal.min(), vmax=test_signal.max(), origin="lower")
title = ax.set_title(f"Test signal: {t[0]}s")
ax.set_xlabel("y")
ax.set_ylabel("x")
def update(frame):
    im.set_data(test_signal[:, :, frame])
    title.set_text(f"Test signal: {t[frame]}s")
    return im, title
ani = FuncAnimation(fig, update, frames=144, interval=100, blit=True)
ani.save('./movie/test.mp4', fps=10)

## 计算小波系数

In [3]:
morlet = Morlet2D1T(
    dx=0.4342,
    dy=0.4342,
    dt=12,
    epsilon=1.0,
    k0=6.0,
    omega0=6.0,
    precision="single",
    backend="gpu",
    device=0,
    boundary=None,
)

In [4]:
k = np.linspace(0.12, 0.18, 8)
theta = np.linspace(-5, 45, 15)
frequency = np.linspace(0.003, 0.004, 8)
omega = 2.0 * np.pi * frequency

In [ ]:
coe_path = morlet.cwt_polar(test_signal, k, theta, omega, output='memmap')

In [6]:
psd_path = morlet.psd_polar(test_signal, k, theta, omega, output='memmap', overwrite=True)

PSD polar:   0%|          | 0/12 [00:00<?, ?block/s]

In [7]:
psd = np.load(psd_path, mmap_mode="r", allow_pickle=False)

In [9]:
psd_xyt = np.trapezoid(psd, x=omega, axis=-1)
psd_xyt = np.trapezoid(psd_xyt, x=theta, axis=-1)
psd_xyt = np.trapezoid(psd_xyt, x=k, axis=-1)

In [ ]:
np.save('./data/psd_xyt.npy', psd_xyt)
psd_xyt = np.load('./data/psd_xyt.npy')

In [9]:
fig, ax = plt.subplots(figsize=(6, 8))
im = ax.imshow(psd_xyt[:, :, 0], cmap='RdBu_r', extent=(y[0], y[-1], x[0], x[-1]), vmin=psd_xyt.min(), vmax=psd_xyt.max(), origin="lower")
title = ax.set_title(f"psd_xyt: {t[0]}s")
ax.set_xlabel("x")
ax.set_ylabel("y")
def update(frame):
    im.set_data(psd_xyt[:, :, frame])
    title.set_text(f"psd_xyt: {t[frame]}s")
    return im, title
ani = FuncAnimation(fig, update, frames=144, interval=100, blit=True)
ani.save('./movie/psd_xyt_test.mp4', fps=10)

<IPython.core.display.Javascript object>